# ⚠️ REQUIRED BEFORE RUNNING — Enable Free GPU

**If you skip this step, the notebook will crash or take 30+ minutes.**

1. Click **Runtime** in the menu above
2. Click **Change runtime type**
3. Set **Hardware accelerator** to **T4 GPU**
4. Click **Save**

Then: **Runtime → Run all** (Ctrl+F9 / Cmd+F9)

---

# TunedAI Labs — Causal Reasoning: 4-Model Comparison

This notebook compares four AI models on the same causal reasoning questions:

| Model | What it is |
|---|---|
| Base Qwen 2.5-7B | An unmodified open-source model |
| GPT-4o | OpenAI's best model (optional — needs API key) |
| Claude 3.5 Sonnet | Anthropic's best model (optional — needs API key) |
| **TunedAI Labs Causal Model** | The same Qwen 2.5-7B, fine-tuned by TunedAI Labs on causal reasoning |

---

## Before you start — one required step

**You must enable a free GPU or the models will load very slowly.**

1. Click **Runtime** in the menu above
2. Click **Change runtime type**
3. Under **Hardware accelerator**, select **T4 GPU**
4. Click **Save**

Then click **Runtime → Run all** (or press Ctrl+F9). The notebook will run automatically.

Loading takes about **2 minutes**. You will see progress messages appear. Scroll down after it finishes to read the results.

---

### About the questions

Every question in this notebook comes from a book or paper published **before AI existed**. The correct answers were worked out by human experts — philosophers, doctors, and statisticians — decades ago. This means the models cannot have simply memorized the answers from the internet.

Sources: John Snow (1854), Ignaz Semmelweis (1847), E.H. Simpson (1951), Bradford Hill (1965), David Hume (1748), R.A. Fisher (1935)
---

### A note on these questions

The six questions below are a representative sample. We tested the TunedAI Labs model across a much wider range of causal reasoning problems — different domains, different structures, different levels of difficulty — and saw consistent results in the same range. The full benchmark score of **96.96% across 10,112 questions** reflects that broader testing. These six are here because they are the most intuitive to read and verify.


---
## Cell 1 of 4 — Install required packages

This installs the software needed to run the models. You will see a lot of text scroll by — that is normal. Wait for it to finish before the next cell runs.

In [ ]:
import subprocess, sys

packages = ['transformers', 'peft', 'accelerate', 'bitsandbytes', 'openai', 'anthropic']

for pkg in packages:
    print(f"Installing {pkg}...", end=" ")
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                       capture_output=True, text=True)
    print("✓" if r.returncode == 0 else f"✗ {r.stderr[:100]}")

print("\nAll packages done.")


---
## Cell 2 of 4 — Optional: Add your API keys

If you want to include **GPT-4o** and **Claude 3.5** in the comparison, paste your API keys below between the quotes.

- OpenAI key: get one at **platform.openai.com** → API Keys
- Anthropic key: get one at **console.anthropic.com** → API Keys

**If you leave these blank, those columns will be skipped.** Base Qwen vs TunedAI Labs still runs with no keys needed.

In [ ]:
OPENAI_API_KEY    = ""   # paste your OpenAI key here (optional)
ANTHROPIC_API_KEY = ""   # paste your Anthropic key here (optional)

---
## Cell 3 of 4 — Load the models

This downloads and loads two versions of the same AI model:
- The original unmodified version (Base Qwen 2.5-7B)
- TunedAI Labs fine-tuned version with causal reasoning training

**This takes about 2 minutes.** You will see messages like "Loading tokenizer", "Loading base model", "Loading TunedAI Labs adapter". Wait for the green checkmark ✓ before continuing.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import openai, anthropic

# Check GPU
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU found — set runtime to A100 via Runtime → Change runtime type")

BASE_MODEL   = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER_REPO = "tunedailabs/causal-reasoning-qwen-7b"

print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
print("✓ Tokenizer loaded")

print("\nLoading base model (~90 sec on A100)...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="cuda:0",
    trust_remote_code=True,
)
print("✓ Base model loaded")

print("\nLoading TunedAI Labs causal adapter...")
tuned_model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
tuned_model.eval()
print("✓ Adapter loaded")

oai_client = openai.OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
ant_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None

print("\n✓ All models ready. Scroll down to see the results.")


---
## Cell 4 of 4 — Helper functions

This sets up the comparison engine. It runs automatically — nothing to change here.

In [ ]:
import re, json, os, torch

SYSTEM = ("You are a careful causal reasoning expert. "
          "When asked a yes/no question about causation, state YES or NO clearly, "
          "then explain your reasoning precisely.")
RESULTS_FILE = "/content/tunedai_results.json"

_keywords = [
    # Q1 — NO — Q1 — Simpson's Paradox: Which Treatment?
    [['no'], ['stone size', 'subgroup', 'confound', 'stratif'], ['treatment a', 'better in every', 'simpson']],
    # Q2 — NO — Q2 — Altitude and Cholera: Does Altitude
    [['no'], ['proxy', 'confounder', 'water', 'not the cause'], ['snow', 'farr wrong', 'altitude correlat']],
    # Q3 — NO — Q3 — Praise and Performance: Does Praise
    [['no'], ['regression to the mean', 'statistical', 'luck', 'noise'], ['control', 'coincidence', 'not causal', 'natural']],
    # Q4 — NO — Q4 — Bomber Armor: Should We Armor the W
    [['no'], ['survivorship', "didn't return", 'shot down', 'missing'], ['engines', 'cockpit', 'armor where no holes']],
    # Q5 — NO — Q5 — City Hospitals: Do They Cause Highe
    [['no'], ['severity', 'case mix', 'confound', 'sicker patients'], ['fair comparison', 'same condition', 'not the hospital']],
    # Q6 — NO — Q6 — Factory Lighting: Does Lighting Cau
    [['no'], ['observed', 'being watched', 'attention', 'hawthorne'], ['not lighting', 'awareness', 'behavior changed']],
    # Q7 — NO — Q7 — Koch Step 1: Does Finding a Bacteri
    [['no'], ['correlation', 'not sufficient', 'step 3', 'introduce', 'experiment'], ['healthy carrier', 'causation requires', 'intervention']],
    # Q8 — NO — Q8 — Berkeley Admissions: Does the Overa
    [['no'], ['simpson', 'department', 'applied to competitive', 'confounder'], ['not discrimination', 'aggregate mislead', 'selectivity']],
    # Q9 — NO — Q9 — Two Fires: By the But-For Test, Did
    [['no'], ['but-for fails', 'overdetermin', 'both fires', 'test breaks'], ['simultaneous', 'sufficient', 'neither passes']],
    # Q10 — NO — Q10 — Spontaneous Generation: Does Air C
    [['no'], ['particles', 'not air', 'contamination', 'dust'], ['swan-neck', 'curve', 'separates', 'air alone']],
    # Q11 — NO — Q11 — Drug Trial: Does the Observational
    [['no'], ['confound', 'selection', 'sicker', 'indication'], ['randomize', 'not prove', 'bias', 'physician choice']],
    # Q12 — NO — Q12 — Brewery Workers: Does Living Near 
    [['no'], ['drinking', 'not proximity', 'water', 'ingestion'], ['negative control', 'brewery well', 'proximity not sufficient']],
    # Q13 — NO — Q13 — Ice Cream and Drowning: Does Ice C
    [['no'], ['common cause', 'temperature', 'summer', 'weather', 'confound'], ['not causal', 'heat', 'swimming', 'third variable']],
    # Q14 — NO — Q14 — Garlic and Scurvy: Does Garlic Cur
    [['no'], ['no recovery', 'garlic failed', 'not effective', 'citrus only'], ['controlled trial', 'lind', 'simultaneous comparison']],
    # Q15 — NO — Q15 — Hospitals and Sickness: Do Hospita
    [['no'], ['reverse causation', 'sick people go', 'selection', 'already ill'], ['not the hospital', 'direction wrong', 'cause and effect reversed']],
    # Q16 — YES — Q16 — Semmelweis: Does Handwashing Preve
    [['yes'], ['yes', 'handwashing', 'prevent', 'caused the drop'], ['intervention', 'mandate', 'before and after', 'autopsy']],
    # Q17 — YES — Q17 — Goldberger: Is Pellagra Caused by 
    [['yes'], ['yes', 'diet', 'deficiency', 'not infectious'], ['induced', 'eliminated', 'self-injection', 'ruled out infection']],
    # Q18 — YES — Q18 — Snow's Pump Handle: Did the Pump C
    [['yes'], ['yes', 'pump caused', 'water', 'contaminated'], ['intervention', 'handle removal', 'cases fell', 'established causation']],
    # Q19 — YES — Q19 — Lind's Trial: Did Citrus Cause the
    [['yes'], ['yes', 'citrus', 'caused', 'recovery'], ['simultaneous', 'controlled', 'only treatment', 'all else equal']],
    # Q20 — YES — Q20 — Bradford Hill: Does Smoking Cause 
    [['yes'], ['yes', 'smoking causes', 'causation', 'cause lung cancer'], ['dose-response', 'temporality', 'consistency', 'totality']],
    # Q21 — YES — Q21 — Doll and Hill: Does Quitting Smoki
    [['yes'], ['yes', 'quitting reduces', 'risk decreases', 'lower risk'], ['dose-response', 'longer quit', 'reversible', 'gradient']],
    # Q22 — YES — Q22 — Snow's Natural Experiment: Does Wa
    [['yes'], ['yes', 'water source', 'causes', 'contaminated water'], ['natural experiment', 'no choice', 'random', 'rules out self-selection']],
    # Q23 — YES — Q23 — Jenner: Did Cowpox Inoculation Pre
    [['yes'], ['yes', 'cowpox protected', 'inoculation caused', 'prevented smallpox'], ['intervention', 'deliberate', 'controlled exposure', 'repeated']],
    # Q24 — YES — Q24 — Streptomycin Trial: Did Streptomyc
    [['yes'], ['yes', 'streptomycin caused', 'randomiz', 'causal'], ['sealed envelopes', 'controlled', 'both groups equal', 'rct']],
    # Q25 — YES — Q25 — Lead Pipes: Did Lead Pipes Cause t
    [['yes'], ['yes', 'lead pipes caused', 'lead', 'single difference'], ['method of difference', 'identical except', 'only variable', 'mill']],
    # Q26 — NO — Q26 — Tutoring and Test Scores: Does the
    [['no'], ['regression to the mean', 'statistical', 'no control', 'not prove'], ['low scorers improve naturally', 'luck', 'need control group']],
    # Q27 — NO — Q27 — Doctors and Heart Disease: Do More
    [['no'], ['confound', 'wealth', 'diagnosis', 'life expectancy', 'third variable'], ['not causal', 'survive longer', 'detected more', 'wealthier countries']],
    # Q28 — NO — Q28 — Placebo Response: Does 35% Improve
    [['no'], ['real', 'biological', 'endorphin', 'expectation', 'not fake'], ['placebo is real', 'mechanism', 'natural course', 'not psychosomatic']],
    # Q29 — YES — Q29 — Benzene and Leukemia: Does Benzene
    [['yes'], ['yes', 'benzene causes', 'causation', 'dose-response'], ['gradient', 'concomitant variation', 'smooth', 'each level']],
    # Q30 — NO — Q30 — The Rooster and the Sun: Does the 
    [['no'], ['common cause', 'earth rotation', 'no', 'not causal'], ['conjunction not causation', 'correlation', 'hume limit', 'coincidence']],
]

def _score(answer, q_idx):
    if q_idx >= len(_keywords): return 0, 3
    ans = answer.lower()
    hits = sum(1 for grp in _keywords[q_idx] if any(kw in ans for kw in grp))
    return hits, len(_keywords[q_idx])

_results = {}
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        _results = json.load(f)
    print(f"Loaded {len(_results)} saved results from {RESULTS_FILE}")

_q_counter = [0]

def ask_local(question, use_adapter=True, max_new_tokens=300):
    if use_adapter:
        tuned_model.enable_adapter_layers()
    else:
        tuned_model.disable_adapter_layers()
    messages = [{"role":"system","content":SYSTEM},{"role":"user","content":question}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = tuned_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

def ask_gpt4(question):
    if not oai_client:
        return "[Skipped — no OpenAI API key]"
    r = oai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role":"system","content":SYSTEM},{"role":"user","content":question}],
        max_tokens=300
    )
    return r.choices[0].message.content.strip()

def ask_claude(question):
    if not ant_client:
        return "[Skipped — no Anthropic API key]"
    r = ant_client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=300,
        system=SYSTEM,
        messages=[{"role":"user","content":question}]
    )
    return r.content[0].text.strip()

def compare_all(question, source="", note=""):
    q_idx = _q_counter[0]
    _q_counter[0] += 1
    key = str(q_idx)

    if key in _results:
        r = _results[key]
        b_h, b_t = r["base_score"]; t_h, t_t = r["tuned_score"]
        print(f"[Q{q_idx+1} already done — Base {b_h}/{b_t} · TunedAI {t_h}/{t_t}]")
        return

    SEP = "=" * 70
    DIV = "-" * 70
    print(SEP)
    if source: print(f"SOURCE : {source}")
    if note:   print(f"ANSWER : {note}")
    print(SEP)
    print(f"QUESTION:\n{question}\n")

    base_ans   = ask_local(question, use_adapter=False)
    gpt_ans    = ask_gpt4(question)
    claude_ans = ask_claude(question)
    tuned_ans  = ask_local(question, use_adapter=True)

    for label, ans in [
        ("BASE QWEN 2.5-7B (untuned)", base_ans),
        ("GPT-4o", gpt_ans),
        ("CLAUDE 3.5 SONNET", claude_ans),
        ("TUNEDAI CAUSAL MODEL ★", tuned_ans),
    ]:
        print(DIV)
        print(f"[ {label} ]")
        print(DIV)
        print(ans)
        print()

    b_hits, b_tot = _score(base_ans, q_idx)
    t_hits, t_tot = _score(tuned_ans, q_idx)
    star = " ★" if t_hits > b_hits else (" ↓" if t_hits < b_hits else "")
    print(f"  Score → Base {b_hits}/{b_tot}  ·  TunedAI {t_hits}/{t_tot}{star}")

    _results[key] = {
        "base_ans": base_ans, "tuned_ans": tuned_ans,
        "base_score": [b_hits, b_tot], "tuned_score": [t_hits, t_tot]
    }
    with open(RESULTS_FILE, "w") as f:
        json.dump(_results, f, indent=2)

print("✓ Ready — running all tests now...")


---
# Results

Each section below presents a scenario from a pre-AI historical source.
Every question has a correct YES or NO answer established by experts decades before computers existed.

**Scoring:** 3 points per question (correct yes/no + key reasoning concepts) × 30 questions = 90 points total.
Results save automatically — if Colab restarts, run all cells again and completed questions are skipped.

---
## Test 1 — Simpson's Paradox: Which Treatment?
**Source:** E.H. Simpson, The Interpretation of Interaction in Contingency Tables, 1951

Treatment A outperforms Treatment B in every subgroup — but appears worse in the combined total. A doctor sees only the combined numbers.

In [ ]:
compare_all(
    question=(
        "A hospital compares two kidney stone treatments:\n\nSmall stones:  Treatment A 93% success  |  Treatment B 87% success\nLarge stones:  Treatment A 73% success  |  Treatment B 69% success\nCombined total: Treatment A 78%         |  Treatment B 83%\n\nThe doctor has not yet run tests to determine the patient's stone size.\n\nBased on the combined data, Treatment B shows higher overall success.\n\nShould the doctor choose Treatment B for this patient?\n\nAnswer YES or NO, then explain your reasoning."
    ),
    source='E.H. Simpson, The Interpretation of Interaction in Contingen',
    note='NO. Treatment A is better for every subgroup. The combined total is misleading because large-stone patients (harder to treat) were disproportionately assigned to Treatment A. Stone size is a confounder.'
)

---
## Q2 — Altitude and Cholera: Does Altitude Cause It?
**Source:** William Farr, Letter to the Registrar General, 1852

Farr found a near-perfect correlation between low altitude and cholera deaths across London parishes. He concluded bad air at low elevations caused the disease.

In [ ]:
compare_all(
    question=(
        "William Farr's 1852 London mortality data:\n\n0-20 feet above sea level:   102 cholera deaths per 10,000\n20-40 feet:                   65 deaths per 10,000\n40-60 feet:                   34 deaths per 10,000\n60-100 feet:                  24 deaths per 10,000\nAbove 100 feet:               17 deaths per 10,000\n\nThe correlation between altitude and cholera deaths is nearly perfect.\n\nDoes low altitude cause cholera?\n\nAnswer YES or NO, then explain your reasoning."
    ),
    source='William Farr, Letter to the Registrar General, 1852',
    note='NO. Low altitude is a proxy for water source. Low-lying areas used Thames water drawn below sewage outfalls. Snow proved water contamination was the true cause. Altitude correlated perfectly but was not causal.'
)

---
## Q3 — Praise and Performance: Does Praise Cause Worse Landings?
**Source:** Francis Galton, Regression Towards Mediocrity in Hereditary Stature, 1886

A flight instructor noticed that praising an unusually good landing was followed by a worse one, and criticizing a bad landing was followed by a better one. He concluded praise causes worse performance.

In [ ]:
compare_all(
    question=(
        'A flight instructor tracks pilot performance over hundreds of training sessions.\n\nPattern observed:\n- After an unusually GOOD landing (top 10%) that the instructor praised: the next landing was worse on average.\n- After an unusually BAD landing (bottom 10%) that the instructor criticized: the next landing was better on average.\n\nThe instructor concludes: praise causes worse performance; criticism causes better performance.\n\nDoes praise cause the worse performance observed after unusually good landings?\n\nAnswer YES or NO, then explain your reasoning.'
    ),
    source='Francis Galton, Regression Towards Mediocrity in Hereditary ',
    note="NO. Regression to the mean. Unusually good or bad performances include a luck component. The next measurement naturally moves back toward average regardless of the instructor's response. The instructor is attributing cause to a statistical artifact."
)

---
## Q4 — Bomber Armor: Should We Armor the Wings?
**Source:** Abraham Wald, Statistical Research Group, Columbia University, 1943

Engineers mapped bullet holes on returning Allied bombers. Holes were clustered on wings and fuselage, almost none on engines and cockpit. The military planned to armor where the holes were.

In [ ]:
compare_all(
    question=(
        "World War II. Engineers examined hundreds of Allied bombers that returned from combat missions and mapped bullet hole locations:\n\nWings and tail:         many bullet holes\nFuselage body:          many bullet holes\nEngines and cockpit:    very few bullet holes\n\nThe military's plan: add armor to the wings and fuselage, since that is where the damage concentrates.\n\nShould engineers add armor to the wings and fuselage where the bullet holes are concentrated?\n\nAnswer YES or NO, then explain your reasoning."
    ),
    source='Abraham Wald, Statistical Research Group, Columbia Universit',
    note='NO. Survivorship bias. The planes with holes in engines and cockpit never returned — they were shot down. The sample only includes survivors. Bullet holes on wings and fuselage are survivable damage. The engines and cockpit are where armor is needed most.'
)

---
## Q5 — City Hospitals: Do They Cause Higher Mortality?
**Source:** Florence Nightingale, Notes on Hospitals, 1863

Nightingale found that London teaching hospitals had death rates three times higher than rural cottage hospitals. A journalist argued patients should avoid city hospitals.

In [ ]:
compare_all(
    question=(
        'English hospital mortality statistics, 1861:\n\nLondon teaching hospitals:  90 deaths per 1,000 patients\nRural cottage hospitals:    30 deaths per 1,000 patients\n\nAdditional facts:\n- London teaching hospitals admit emergency cases, trauma, advanced disease, major surgery\n- Rural cottage hospitals treat minor illness, childbirth complications, and recovery cases\n- The two types of hospitals almost never serve the same patients\n\nDoes going to a city hospital cause higher mortality risk compared to a rural hospital?\n\nAnswer YES or NO, then explain your reasoning.'
    ),
    source='Florence Nightingale, Notes on Hospitals, 1863',
    note='NO. Case severity is a confounder. City hospitals treat patients who would die anywhere without intervention. A fair comparison would compare outcomes for patients with identical conditions across both settings.'
)

---
## Q6 — Factory Lighting: Does Lighting Cause Productivity?
**Source:** Elton Mayo, Hawthorne Works studies, Western Electric Company, 1924-1932

Researchers tested the effect of lighting on factory worker productivity. Every change they made — including reversals — improved productivity.

In [ ]:
compare_all(
    question=(
        'Hawthorne Works factory, Illinois. Researchers studied whether lighting affects worker productivity.\n\nResults:\n- Increased lighting from standard to bright: productivity increased\n- Decreased lighting back to standard: productivity still increased\n- Decreased lighting below standard: productivity still increased\n- Removed all changes, returned to original: productivity remained elevated\n\nEvery change in lighting — up or down — was followed by improved productivity.\n\nDoes the lighting level cause the productivity improvements observed in these studies?\n\nAnswer YES or NO, then explain your reasoning.'
    ),
    source='Elton Mayo, Hawthorne Works studies, Western Electric Compan',
    note='NO. Workers improved because they knew they were being studied — not because of lighting changes. The intervention was observation itself, not lighting. This is now called the Hawthorne Effect.'
)

---
## Q7 — Koch Step 1: Does Finding a Bacterium Prove It Causes Disease?
**Source:** Robert Koch, tuberculosis research, 1876-1884

Koch found Mycobacterium tuberculosis in every tuberculosis patient he examined. A colleague argued this finding alone proved the bacterium caused the disease.

In [ ]:
compare_all(
    question=(
        'Robert Koch examines lung tissue from 100 tuberculosis patients.\nHe finds Mycobacterium tuberculosis in every single case.\nHe examines 50 healthy individuals. The bacterium is absent in all of them.\n\nA colleague argues: the bacterium is present in 100% of sick patients and 0% of healthy ones. This perfect correlation proves the bacterium causes tuberculosis.\n\nDoes finding the bacterium in every sick patient and no healthy patients prove that it causes tuberculosis?\n\nAnswer YES or NO, then explain your reasoning.'
    ),
    source='Robert Koch, tuberculosis research, 1876-1884',
    note="NO. Correlation is not causation. The bacterium could be a consequence of the disease, or both could share a common cause. Koch's own framework requires Step 3: introduce the isolated organism to a healthy individual and reproduce the disease."
)

---
## Q8 — Berkeley Admissions: Does the Overall Rate Prove Discrimination?
**Source:** P.J. Bickel, E.A. Hammel, J.W. O'Connell, Science, 1975

UC Berkeley was sued for sex discrimination based on overall admission rates. Department-level data told a completely different story.

In [ ]:
compare_all(
    question=(
        'UC Berkeley graduate admissions, Fall 1973:\n\nOverall: Men 44% admitted, Women 35% admitted\n\nDepartment-level data (top 6 departments):\nDept A: Men 62%, Women 82%\nDept B: Men 63%, Women 68%\nDept C: Men 37%, Women 34%\nDept D: Men 33%, Women 35%\nDept E: Men 28%, Women 24%\nDept F: Men 6%,  Women 7%\n\nWomen applied in much higher proportions to Departments E and F (the most competitive, lowest admission rates). Men applied in higher proportions to Departments A and B (less competitive).\n\nDoes the overall admission rate difference (44% vs 35%) prove the university discriminated against women?\n\nAnswer YES or NO, then explain your reasoning.'
    ),
    source="P.J. Bickel, E.A. Hammel, J.W. O'Connell, Science, 1975",
    note="NO. Simpson's Paradox. Women self-selected into more competitive departments at higher rates. Within each department, women were admitted at equal or higher rates. The aggregate gap is explained entirely by application patterns, not bias in admissions decisions."
)

---
## Q9 — Two Fires: By the But-For Test, Did Fire 1 Cause the Destruction?
**Source:** David Hume, An Enquiry Concerning Human Understanding, 1748; Hart and Honore, 1959

Hume proposed: if the first event had NOT happened, and the second would still have occurred, then the first event did not cause the second. Two simultaneous fires test this rule.

In [ ]:
compare_all(
    question=(
        "Two forest fires start independently and simultaneously on opposite sides of a valley.\nEither fire alone would be sufficient to burn the farmhouse in the valley.\nThe fires merge and the farmhouse burns.\n\nApplying Hume's but-for test to Fire 1:\nIf Fire 1 had NOT started, would the farmhouse still have burned?\nYes — Fire 2 would have burned it anyway.\n\nBy Hume's but-for test: did Fire 1 cause the farmhouse to burn?\n\nAnswer YES or NO, then explain your reasoning."
    ),
    source='David Hume, An Enquiry Concerning Human Understanding, 1748;',
    note="NO — by the but-for test. But the but-for test gives the wrong answer here. Both fires are genuine causes. This is overdetermination: multiple sufficient causes that each individually break the standard test. Hume's test fails when two independent causes are each sufficient to produce the effect."
)

---
## Q10 — Spontaneous Generation: Does Air Cause Life to Appear?
**Source:** Louis Pasteur, Memoire sur les corpuscles organises, 1859

The spontaneous generation debate held that something in air could generate life from broth. Pasteur's swan-neck flask separated 'air' from 'particles in air' to test this.

In [ ]:
compare_all(
    question=(
        "Pasteur's experiment:\n\nFlask A (sealed): broth boiled, sealed from all air. Stays clear indefinitely.\nFlask B (open neck): broth boiled, open to air. Microorganisms appear within days.\nFlask C (swan-neck): broth boiled, long curved tube open to air but curved to trap particles. Stays clear for months despite being open to air.\n\nThen Pasteur tipped Flask C so broth touched the dust trapped in the curve.\nMicroorganisms appeared in Flask C within days.\n\nDoes air itself cause microorganisms to appear in broth?\n\nAnswer YES or NO, then explain your reasoning."
    ),
    source='Louis Pasteur, Memoire sur les corpuscles organises, 1859',
    note='NO. Flask C proves air alone does not cause microorganisms — it stayed clear while open to air. The particles trapped in the curve, not the air itself, carried the microorganisms. Life comes from pre-existing life via contamination, not spontaneous generation from air.'
)

---
## Q11 — Drug Trial: Does the Observational Study Prove the Drug Causes Harm?
**Source:** R.A. Fisher, The Design of Experiments, 1935

Doctors gave a new drug to their sickest patients. The drug group had worse outcomes. A critic argued the study proved the drug was harmful.

In [ ]:
compare_all(
    question=(
        'A hospital tests a new medication:\n\nMethod: Doctors choose which patients receive the drug based on clinical judgment.\nDoctors tend to give the drug to patients who are more severely ill.\n\nResults:\n- Drug group: 35% mortality rate\n- No-drug group: 18% mortality rate\n\nThe drug group has nearly double the mortality of the no-drug group.\n\nDoes this observational study prove the drug causes higher mortality?\n\nAnswer YES or NO, then explain your reasoning.'
    ),
    source='R.A. Fisher, The Design of Experiments, 1935',
    note="NO. Confounding by indication. Doctors gave the drug to sicker patients, so the drug group was already at higher risk of dying. The drug group's higher mortality reflects patient severity, not drug harm. Only a randomized trial with equal group severity could establish causation."
)

---
## Q12 — Brewery Workers: Does Living Near the Pump Cause Cholera?
**Source:** John Snow, On the Mode of Communication of Cholera, 2nd edition, 1855

During the Broad Street outbreak, brewery workers lived and worked next to the contaminated pump but had zero deaths. Snow used this as a negative control.

In [ ]:
compare_all(
    question=(
        'Broad Street cholera outbreak, London, 1854.\n\nA brewery stood directly on Broad Street, next to the contaminated pump.\nThe brewery employed over 70 workers who were on the premises throughout the outbreak.\n\nDeaths among surrounding residents: 500+ in 10 days.\nDeaths among brewery workers: 0.\n\nInvestigation: The brewery had its own private well. Workers were given a daily ration of malt liquor. None of the workers drank from the Broad Street pump.\n\nDoes living or working near the contaminated Broad Street pump cause cholera?\n\nAnswer YES or NO, then explain your reasoning.'
    ),
    source='John Snow, On the Mode of Communication of Cholera, 2nd edit',
    note='NO. Proximity to the pump did not cause cholera — drinking its water did. The brewery workers were meters away from the pump for weeks and had zero deaths because they never drank from it. This negative control demonstrates that the transmission route is consumption, not proximity or air.'
)

---
## Q13 — Ice Cream and Drowning: Does Ice Cream Cause Drowning?
**Source:** Classic confounding example — documented in epidemiology textbooks from the 1950s onward

Monthly data across 10 years showed a nearly perfect correlation between ice cream sales and drowning deaths. Both peak in summer, both drop in winter.

In [ ]:
compare_all(
    question=(
        'Monthly statistics, American coastal cities, 1950-1960:\n\nJanuary:  Low ice cream sales, low drowning deaths\nApril:    Rising ice cream sales, rising drowning deaths\nJuly:     Peak ice cream sales, peak drowning deaths\nOctober:  Falling ice cream sales, falling drowning deaths\n\nThe correlation between ice cream sales and drowning deaths across all months is r = 0.97.\n\nDoes eating ice cream cause drowning deaths?\n\nAnswer YES or NO, then explain your reasoning.'
    ),
    source='Classic confounding example — documented in epidemiology tex',
    note='NO. A common cause — hot weather — drives both. Heat increases both ice cream consumption and swimming activity (which increases drowning risk). Ice cream and drowning are correlated because they share a cause, not because one causes the other.'
)

---
## Q14 — Garlic and Scurvy: Does Garlic Cure Scurvy?
**Source:** James Lind, A Treatise of the Scurvy, 1753

Before Lind's trial, garlic was a popular folk remedy for scurvy. Sailors given garlic sometimes recovered. Lind's controlled trial showed only citrus worked.

In [ ]:
compare_all(
    question=(
        "Historical belief, 1740s: Garlic prevents and cures scurvy. Ships regularly stocked garlic. Ships' logs recorded cases where sailors given garlic recovered from scurvy.\n\nJames Lind's 1747 trial aboard HMS Salisbury:\n12 sailors with advanced scurvy. Same base diet. 6 different supplement pairs:\n\nPair 1 — Cider:            No meaningful recovery\nPair 2 — Sulfuric acid:    No meaningful recovery\nPair 3 — Vinegar:          No meaningful recovery\nPair 4 — Seawater:         No meaningful recovery\nPair 5 — Oranges & lemons: Full recovery within 6 days\nPair 6 — Garlic paste:     No meaningful recovery\n\nDoes garlic cure scurvy?\n\nAnswer YES or NO, then explain your reasoning."
    ),
    source='James Lind, A Treatise of the Scurvy, 1753',
    note="NO. Lind's controlled trial showed garlic produced no meaningful recovery while citrus produced full recovery in the same conditions over the same time period. Earlier sailors' recovery was coincidental — scurvy can temporarily improve with rest, and they may have received citrus at port."
)

---
## Q15 — Hospitals and Sickness: Do Hospitals Make People Sick?
**Source:** Classic reverse causation problem — foundational in epidemiology

Hospitals have far more sick people per square mile than any other type of building. A naive analysis of this association produces a dangerous conclusion.

In [ ]:
compare_all(
    question=(
        'Observation: In every city studied, the buildings with the highest concentration of sick people are hospitals. People who enter hospitals are significantly more likely to be ill than people who do not enter hospitals. In fact, nearly 100% of hospital inpatients are ill.\n\nA researcher notes: People who go to hospitals become sick at dramatically higher rates than people who stay home. The association is strong, consistent, and found in every country.\n\nDoes going to a hospital cause people to become sick?\n\nAnswer YES or NO, then explain your reasoning.'
    ),
    source='Classic reverse causation problem — foundational in epidemio',
    note='NO. Reverse causation. Sick people go to hospitals — hospitals do not make people sick. The causal direction runs from illness to hospital admission, not from hospital admission to illness. This is a textbook case of mistaking the direction of causality.'
)

---
## Q16 — Semmelweis: Does Handwashing Prevent Childbed Fever Deaths?
**Source:** Ignaz Semmelweis, Vienna General Hospital records, 1847

Semmelweis mandated chlorinated lime handwashing for all doctors before entering the maternity ward. Mortality collapsed immediately.

In [ ]:
compare_all(
    question=(
        "Vienna General Hospital, 1847:\n\nBefore handwashing mandate:\n- First Clinic (doctors, who came directly from performing autopsies): 10% monthly mortality rate\n- Second Clinic (midwives, who did not perform autopsies): 2% monthly mortality rate\n\nSemmelweis required all doctors to wash hands with chlorinated lime solution before entering the maternity ward.\n\nAfter handwashing mandate:\n- First Clinic mortality: dropped to 1.5% — below the midwives' ward\n- Second Clinic mortality: remained at 2% (no change — midwives already did not perform autopsies)\n\nDoes handwashing with chlorinated lime prevent childbed fever deaths in the doctors' ward?\n\nAnswer YES or NO, then explain your reasoning."
    ),
    source='Ignaz Semmelweis, Vienna General Hospital records, 1847',
    note="YES. The before-and-after intervention shows a clear causal effect. Mortality dropped immediately after the mandate and only in the ward where the intervention occurred. The midwives' ward, used as a natural control, showed no change — ruling out external factors."
)

---
## Q17 — Goldberger: Is Pellagra Caused by Diet Deficiency?
**Source:** Joseph Goldberger, U.S. Public Health Service, 1915

Pellagra was believed to be infectious. Goldberger improved diets and eliminated it, induced it by restricting diet, and injected himself with patient material without getting sick.

In [ ]:
compare_all(
    question=(
        "Joseph Goldberger's three-part investigation:\n\nStudy 1 — Diet improvement: Goldberger gave a Mississippi orphanage a more varied diet. Pellagra cases dropped from 68 to 0 within months.\n\nStudy 2 — Diet induction: Goldberger fed 11 Mississippi prisoners a corn-heavy restricted diet. 6 of 11 developed pellagra within 5 months.\n\nStudy 3 — Infection test: Goldberger and 16 volunteers injected themselves with blood, swabbed nasal passages with secretions, and swallowed skin material from active pellagra patients. None developed pellagra.\n\nIs pellagra caused by dietary deficiency rather than an infectious agent?\n\nAnswer YES or NO, then explain your reasoning."
    ),
    source='Joseph Goldberger, U.S. Public Health Service, 1915',
    note='YES. All three studies point to dietary deficiency. Diet improvement eliminated it. Diet restriction induced it. Deliberate exposure to patient material produced no infection. The germ theory is ruled out; dietary cause is established.'
)

---
## Q18 — Snow's Pump Handle: Did the Pump Cause the Outbreak?
**Source:** John Snow, On the Mode of Communication of Cholera, 2nd edition, 1855

Snow removed the handle of the Broad Street pump at the height of the 1854 cholera outbreak. The local epidemic subsided.

In [ ]:
compare_all(
    question=(
        'London, September 1854. Cholera outbreak centered on Broad Street.\n\nSnow mapped every cholera death in the area. Deaths clustered within a few hundred yards of the Broad Street water pump. Streets served by other pumps had far fewer deaths.\n\nSnow persuaded the local Board of Guardians to remove the handle of the Broad Street pump, rendering it unusable.\n\nWithin days, new cholera cases in the Broad Street district fell sharply.\nSurrounding districts without pump handle removal continued to have cases.\n\nDid the contaminated Broad Street pump cause the local cholera outbreak?\n\nAnswer YES or NO, then explain your reasoning.'
    ),
    source='John Snow, On the Mode of Communication of Cholera, 2nd edit',
    note='YES. The pump handle removal is an intervention — Snow controlled what changed. Cases fell in the affected district immediately after removal, while surrounding areas without intervention continued to have cases. This is causal evidence, not mere correlation.'
)

---
## Q19 — Lind's Trial: Did Citrus Cause the Scurvy Recovery?
**Source:** James Lind, A Treatise of the Scurvy, 1753

Lind gave 12 sailors with scurvy 6 different treatments simultaneously. Only the citrus pair recovered. This was the first controlled clinical trial in history.

In [ ]:
compare_all(
    question=(
        '1747, HMS Salisbury. 12 sailors with advanced scurvy. Same base diet throughout.\nSix treatment pairs, each given simultaneously for the same duration:\n\nPair 1 — Cider (1 quart/day):             No recovery\nPair 2 — Sulfuric acid (25 drops/day):     No recovery\nPair 3 — Vinegar (6 spoonfuls/day):        No recovery\nPair 4 — Seawater (half pint/day):         No recovery\nPair 5 — Oranges and lemons (2 and 1/day): Full recovery within 6 days\nPair 6 — Garlic/mustard paste:             No recovery\n\nAll 12 sailors had the same disease severity, same base diet, same ship conditions, tested simultaneously.\n\nDid citrus cause the scurvy recovery in Pair 5?\n\nAnswer YES or NO, then explain your reasoning.'
    ),
    source='James Lind, A Treatise of the Scurvy, 1753',
    note='YES. Simultaneous comparison under identical conditions isolates the citrus as the cause. All other treatments failed. The simultaneous design rules out time, rest, and other factors. This is the defining feature of a controlled experiment.'
)

---
## Q20 — Bradford Hill: Does Smoking Cause Lung Cancer?
**Source:** Bradford Hill, The Environment and Disease: Association or Causation?, 1965

No randomized trial of smoking was ever run. Critics argued this meant only correlation could be claimed. Bradford Hill argued the totality of evidence crossed the threshold for causal inference.

In [ ]:
compare_all(
    question=(
        'Evidence accumulated by 1965:\n\n1. Smokers develop lung cancer at 9-10 times the rate of non-smokers\n2. Heavy smokers develop cancer at higher rates than light smokers (dose-response)\n3. The association has been replicated across 30+ countries and multiple research teams\n4. Lung cancer develops 15-20 years after a person starts smoking — never before\n5. Cigarette smoke contains known chemical carcinogens\n6. When smoking rates rose in men after WWI, lung cancer rates rose 20 years later\n7. Doctors who quit smoking have lower lung cancer rates than those who continue\n\nNo randomized controlled trial was conducted — you cannot force people to smoke for decades.\n\nDoes smoking cause lung cancer?\n\nAnswer YES or NO, then explain your reasoning.'
    ),
    source='Bradford Hill, The Environment and Disease: Association or C',
    note="YES. The totality of evidence — dose-response gradient, correct temporal order, consistency across countries, biological mechanism, and the effect of quitting — crosses the threshold for causal inference established by Bradford Hill's criteria. The absence of an RCT does not prevent causal conclusions when the evidence is this strong."
)

---
## Q21 — Doll and Hill: Does Quitting Smoking Reduce Lung Cancer Risk?
**Source:** Richard Doll and Bradford Hill, British Medical Journal, 1954

Doll and Hill followed 40,000 British doctors prospectively. Among those who quit smoking, lung cancer rates declined — and continued to fall the longer they had been quit.

In [ ]:
compare_all(
    question=(
        "Doll and Hill's prospective study of 40,000 British doctors, followed from 1951.\n\nLung cancer death rates per 1,000 per year:\n\nNon-smokers:                         0.07\nCurrent smokers (25+ cigarettes/day): 1.66\nFormer smokers, quit less than 5 years ago: 0.90\nFormer smokers, quit 5-10 years ago:  0.51\nFormer smokers, quit 10+ years ago:   0.18\n\nThe longer doctors had been quit, the lower their lung cancer rates — approaching but not reaching the non-smoker rate.\n\nDoes quitting smoking reduce lung cancer risk?\n\nAnswer YES or NO, then explain your reasoning."
    ),
    source='Richard Doll and Bradford Hill, British Medical Journal, 195',
    note='YES. The dose-response of quitting — longer cessation, lower risk — is strong causal evidence. If smoking merely correlated with lung cancer without causing it, quitting would not produce a gradient of declining risk. The reversibility of the effect strengthens the causal claim dramatically.'
)

---
## Q22 — Snow's Natural Experiment: Does Water Source Cause Cholera?
**Source:** John Snow, On the Mode of Communication of Cholera, 2nd edition, 1855

Two water companies served intermingled houses on the same streets. Households had no choice in supplier. Death rates differed by a factor of eight.

In [ ]:
compare_all(
    question=(
        "London, 1854. Two water companies served houses on the same streets, with pipes running side by side:\n\nSouthwark and Vauxhall Company:\n- Drew water from the Thames below all sewage outfalls\n- Deaths per 10,000 houses: 315\n\nLambeth Company:\n- Drew water from the Thames above the city, upstream of all outfalls\n- Deaths per 10,000 houses: 37\n\nCrucially: households had no say in which company supplied them. The two companies' pipes ran down the same streets, serving adjacent houses at random.\n\nDoes water source cause the difference in cholera death rates?\n\nAnswer YES or NO, then explain your reasoning."
    ),
    source='John Snow, On the Mode of Communication of Cholera, 2nd edit',
    note='YES. The random-like assignment of water supplier — households had no choice — eliminates self-selection. Rich and poor, healthy and sick, lived side by side and differed only in water source. The eightfold difference in death rates is attributable to water contamination.'
)

---
## Q23 — Jenner: Did Cowpox Inoculation Prevent Smallpox?
**Source:** Edward Jenner, An Inquiry into the Causes and Effects of the Variolae Vaccinae, 1798

Jenner inoculated an 8-year-old boy with cowpox material, then deliberately exposed him to smallpox six weeks later. The boy did not develop smallpox.

In [ ]:
compare_all(
    question=(
        "Edward Jenner, 1796:\n\nStep 1: Jenner took material from a fresh cowpox pustule on milkmaid Sarah Nelmes.\nStep 2: He scratched the material into the arm of James Phipps, age 8.\nStep 3: James developed mild cowpox and recovered fully over two weeks.\nStep 4: Six weeks later, Jenner scratched material from a smallpox patient into James's arm.\nStep 5: James did not develop smallpox.\n\nJenner repeated this protocol on 23 additional individuals over the following years. In every case, cowpox inoculation was followed by resistance to smallpox challenge.\n\nDid cowpox inoculation cause James Phipps to be protected from smallpox?\n\nAnswer YES or NO, then explain your reasoning."
    ),
    source='Edward Jenner, An Inquiry into the Causes and Effects of the',
    note='YES. Jenner controlled the exposure — he deliberately introduced cowpox, then deliberately challenged with smallpox. The outcome (no smallpox) followed from a controlled intervention, not passive observation. Replication across 23 additional subjects strengthens the causal claim.'
)

---
## Q24 — Streptomycin Trial: Did Streptomycin Reduce TB Mortality?
**Source:** MRC Streptomycin in Tuberculosis Trials Committee, British Medical Journal, 1948

The first properly randomized controlled trial in medicine. TB patients were allocated by sealed envelopes to streptomycin or bed rest alone.

In [ ]:
compare_all(
    question=(
        'Medical Research Council, 1947. Tuberculosis patients allocated by sealed envelopes:\n\nStreptomycin + bed rest group (55 patients):\n- Died within 6 months: 7%\n- Radiological improvement: 51%\n\nBed rest only group (52 patients):\n- Died within 6 months: 27%\n- Radiological improvement: 8%\n\nAllocation was by sealed envelopes opened sequentially — neither doctors nor patients could predict or influence assignment.\n\nDid streptomycin cause the reduction in tuberculosis mortality observed in this trial?\n\nAnswer YES or NO, then explain your reasoning.'
    ),
    source='MRC Streptomycin in Tuberculosis Trials Committee, British M',
    note='YES. Random allocation created groups equal on all measured and unmeasured factors. Any difference in outcomes can therefore be attributed to streptomycin. This is the defining power of randomization — it eliminates confounding by design.'
)

---
## Q25 — Lead Pipes: Did Lead Pipes Cause the Illness?
**Source:** John Stuart Mill, A System of Logic, 1843 — Method of Difference

Two villages, identical in every measurable way, differed only in their water pipe material. One developed a pattern of neurological illness; the other did not.

In [ ]:
compare_all(
    question=(
        "Two English villages, 1845. Investigators confirmed they were identical in:\ndiet, water source, altitude, occupations, population density, and climate.\n\nOne difference: Village A replaced its wooden water pipes with lead pipes in 1839.\nVillage B kept its wooden pipes.\n\nOver the following decade:\nVillage A: rising cases of severe abdominal pain, muscle weakness, and partial paralysis.\nVillage B: no such cases.\n\nMill's Method of Difference: if two situations are identical in every way except one, and different outcomes occur, that one difference is the cause.\n\nDid the lead pipes cause the illness in Village A?\n\nAnswer YES or NO, then explain your reasoning."
    ),
    source='John Stuart Mill, A System of Logic, 1843 — Method of Differ',
    note="YES. By Mill's Method of Difference, the single difference between the two villages — lead pipes — is the cause of the illness. The method requires that all other factors be truly identical, which the investigators confirmed. Lead poisoning (plumbism) produces exactly the symptoms described."
)

---
## Q26 — Tutoring and Test Scores: Does the Score Improvement Prove Tutoring Works?
**Source:** Francis Galton, Regression Towards Mediocrity in Hereditary Stature, 1886 — applied to education

A school gave tutoring to its lowest-scoring 20% of students. Their scores improved on the next test. The principal concluded tutoring was effective.

In [ ]:
compare_all(
    question=(
        "A school identifies its lowest-scoring 20% of students on a standardized test.\nThese students are given two weeks of intensive tutoring.\nOn the next test, the tutored students' scores improve by an average of 12 points.\n\nThe principal concludes: tutoring caused the 12-point improvement.\n\nAn outside evaluator notes: no other students were tracked over the same period, and the students selected were specifically chosen for unusually low scores.\n\nDoes the observed score improvement prove that tutoring caused it?\n\nAnswer YES or NO, then explain your reasoning."
    ),
    source='Francis Galton, Regression Towards Mediocrity in Hereditary ',
    note='NO. Students selected for unusually low scores will tend to score higher on the next test regardless of intervention — this is regression to the mean. Without a control group of equally low-scoring students who did not receive tutoring, the improvement cannot be attributed to tutoring rather than statistical reversion.'
)

---
## Q27 — Doctors and Heart Disease: Do More Doctors Cause More Heart Disease?
**Source:** Classic confounding example in epidemiology

Countries with more doctors per capita have higher rates of cardiovascular disease. A politician used this association to argue against expanding medical training.

In [ ]:
compare_all(
    question=(
        'International health statistics, 1970:\n\nCountries ranked by doctors per 100,000 population:\n\nHigh doctor density (USA, Sweden, West Germany):\n- High cardiovascular disease rates\n- High life expectancy\n\nLow doctor density (Uganda, Bolivia, India):\n- Low cardiovascular disease rates recorded\n- Lower life expectancy\n\nThe correlation between doctor density and cardiovascular disease rates is strong and consistent.\n\nDo more doctors per capita cause higher rates of cardiovascular disease?\n\nAnswer YES or NO, then explain your reasoning.'
    ),
    source='Classic confounding example in epidemiology',
    note='NO. Wealth confounds both. Wealthy countries have more doctors AND longer-lived populations who survive long enough to develop cardiovascular disease. They also have better diagnostic infrastructure to detect it. Low-income countries have lower recorded cardiovascular disease partly because patients die of other causes first and lack diagnostic capacity.'
)

---
## Q28 — Placebo Response: Does 35% Improvement on Sugar Pills Mean the Pain Was Not Real?
**Source:** Henry Beecher, The Powerful Placebo, JAMA, 1955

Beecher found that 35% of patients with chronic pain improved on sugar pills. A doctor argued this proved the pain was psychosomatic and therefore required no treatment.

In [ ]:
compare_all(
    question=(
        "Henry Beecher's 1955 analysis of 15 clinical trials covering 1,082 patients:\n\nAverage placebo response rate: 35% of patients showed measurable improvement from sugar pills alone.\nIn chronic pain studies: placebo response reached 50% or higher in several trials.\nThe effect was consistent across different countries, hospitals, and patient populations.\n\nA physician argues: if 35% of patients get better from sugar pills, their pain was not real — it was in their heads — and therefore these patients need no active treatment.\n\nDoes a 35% placebo response rate prove the patients' pain was not real?\n\nAnswer YES or NO, then explain your reasoning."
    ),
    source='Henry Beecher, The Powerful Placebo, JAMA, 1955',
    note='NO. Placebo response is a real biological phenomenon — it can involve endorphin release, changes in neural processing, and reduction of anxiety that amplifies pain. A high placebo response means the mechanism of improvement is expectation and context, not that the original pain was imaginary. It argues for controlling for placebo in drug trials, not for withholding treatment.'
)

---
## Q29 — Benzene and Leukemia: Does Benzene Exposure Cause Leukemia?
**Source:** John Stuart Mill, A System of Logic, 1843 — Method of Concomitant Variation; occupational health studies

Workers exposed to different levels of benzene show a smooth dose-response gradient in leukemia rates. Each doubling of exposure roughly doubles the risk.

In [ ]:
compare_all(
    question=(
        'Occupational health study of chemical factory workers:\n\nAnnual benzene exposure vs leukemia incidence per 100,000 workers per year:\n\nUnexposed (office workers, same factory):  6 cases\nLow (1-10 ppm, 8 hours/day):             14 cases\nMedium (10-50 ppm):                       31 cases\nHigh (50-200 ppm):                        58 cases\nVery high (200+ ppm):                    105 cases\n\nAll groups are similar in age, smoking rates, diet, and other measured health factors.\nThe dose-response is smooth and consistent across all exposure levels.\n\nDoes benzene exposure cause leukemia?\n\nAnswer YES or NO, then explain your reasoning.'
    ),
    source='John Stuart Mill, A System of Logic, 1843 — Method of Concom',
    note="YES. The smooth dose-response gradient is strong causal evidence — Mill's Method of Concomitant Variation. Each increase in exposure produces a proportional increase in leukemia cases. The comparison group is internal (same factory, same workplace culture), controlling for many confounders. Benzene's mechanism of DNA damage is also well established."
)

---
## Q30 — The Rooster and the Sun: Does the Rooster's Crowing Cause Sunrise?
**Source:** David Hume, An Enquiry Concerning Human Understanding, 1748 — the problem of constant conjunction

Hume observed that we infer causation from constant conjunction — A always preceding B. A rooster crowing before sunrise is a perfect case of constant conjunction that is clearly not causal.

In [ ]:
compare_all(
    question=(
        "A rooster on a farm crows reliably every morning before sunrise.\nIn 1,000 consecutive days of observation:\n- The rooster crowed before sunrise on all 1,000 days\n- Sunrise followed the crowing on all 1,000 days\n- The rooster never crowed and had sunrise fail to occur\n- Sunrise never occurred without the rooster crowing first\n\nHume noted that causation is inferred from constant conjunction — A always comes before B, B always follows A, and they are always observed together.\n\nBy Hume's criterion of constant conjunction, does the rooster's crowing cause the sun to rise?\n\nAnswer YES or NO, then explain your reasoning."
    ),
    source='David Hume, An Enquiry Concerning Human Understanding, 1748 ',
    note="NO. Constant conjunction is not sufficient for causation — it is only correlation over time. Both the crowing and sunrise are caused by the Earth's daily rotation. The rooster responds to light before it breaks the horizon; the sunrise follows from orbital mechanics. Removing the rooster would not prevent sunrise. Hume himself acknowledged this limitation of the conjunction criterion."
)

---
## Final Results — All 30 Questions

Keyword scoring: 3 groups per question × 30 questions = 90 points. Reads from saved file — survives Colab restarts.

In [ ]:
import json, os
RESULTS_FILE = "/content/tunedai_results.json"
if not os.path.exists(RESULTS_FILE):
    print("No results yet — run the test cells above first.")
else:
    with open(RESULTS_FILE) as f: res = json.load(f)
    n = len(res)
    b_total = sum(res[k]["base_score"][0] for k in res)
    t_total = sum(res[k]["tuned_score"][0] for k in res)
    maximum = sum(res[k]["base_score"][1] for k in res)
    print("=" * 70)
    print("FINAL RESULTS — TunedAI Labs Causal Reasoning Test")
    print("30 yes/no causal questions from pre-AI historical sources")
    print("=" * 70)
    print(f"  Base Qwen 2.5-7B (untuned) : {b_total:3d}/{maximum} = {100*b_total/maximum:.1f}%")
    print(f"  TunedAI Causal Model ★     : {t_total:3d}/{maximum} = {100*t_total/maximum:.1f}%")
    diff = 100*t_total/maximum - 100*b_total/maximum
    print(f"  Improvement                : {diff:+.1f} percentage points")
    print("=" * 70)
    print(f"\nCompleted: {n}/30\n")
    print(f"{'Q':<6} {'Base':>6} {'Tuned':>7}")
    print(f"{'----':<6} {'------':>6} {'-------':>7}")
    for i in range(n):
        k = str(i)
        if k not in res: continue
        b_h, b_t = res[k]["base_score"]
        t_h, t_t = res[k]["tuned_score"]
        flag = " ★" if t_h > b_h else (" ↓" if t_h < b_h else "")
        print(f"Q{i+1:<5} {b_h}/{b_t}    {t_h}/{t_t}{flag}")


---
## Try Your Own Question

Replace the text below with any causal reasoning question and run the cell.

In [ ]:
MY_QUESTION = """
Type your causal reasoning question here.
"""

compare_all(MY_QUESTION, source="Custom")

---
## Share What You Saw

Open a [GitHub Issue](https://github.com/tunedailabs/tunedailabs/issues/new) and paste your results.

**TunedAI Labs** — We fine-tune open-source LLMs for real-world reasoning.

[tunedailabs.com](https://tunedailabs.com)